# 01 — Coleta de imagens (Mapillary)
**Auditoria de Calçadas — Kayky Oliveira Schunck (RM 99756)**

Baixa as fotos de calçadas do bairro do Brás (São Paulo) publicadas no Mapillary pelo gabinete da vereadora Renata Falzoni no projeto *Auditoria Cidadã de Calçadas* (nov/2025). Baseado no notebook de apoio da disciplina.

1. Crie um token em mapillary.com/dashboard/developers (permissão READ).
2. Guarde-o nos *Secrets* do Colab com o nome `Mapillary_api`.
3. Rode as células. Saída: `auditoria_calcadas.zip` com as imagens e um CSV de metadados (id, data, lat/lon).

Das 300 fotos sorteadas, 65 foram selecionadas manualmente para rotulagem (ver README).

In [ ]:
import os
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================
# CONFIGURAÇÕES
# ============================================================

# Cole aqui seu token do Mapillary
ACCESS_TOKEN = "COLOQUE_SEU_TOKEN_AQUI"

# Usuário usado pelo projeto original
CREATOR_NAME = "gabinete_falzoni"

# ID do usuário usado no script R original
TARGET_USER_ID = "982503570396388"

# Mesmo período usado atualmente pelo script do repositório
START_DATE = "2025-11-08T00:00:00Z"
END_DATE   = "2025-11-17T23:59:59Z"

# Bounding box do Brás usada no repositório:
# xmin, ymin, xmax, ymax
BBOX = (
    -46.628036,
    -23.549392,
    -46.605377,
    -23.529917
)

# Pasta onde tudo será salvo
OUTPUT_DIR = "auditoria_calcadas"

IMAGES_DIR = os.path.join(OUTPUT_DIR, "imagens")
CSV_PATH = os.path.join(OUTPUT_DIR, "fotos_mapillary.csv")

# Número de divisões da área.
#
# O código original usa 100 x 100.
# Isso gera 10.000 pequenas consultas.
#
# Para começar, recomendo 20 x 20.
# Se perceber que faltaram imagens, aumente.
GRID_X = 20
GRID_Y = 20

# Downloads simultâneos
DOWNLOAD_WORKERS = 10

# Timeout HTTP
TIMEOUT = 30


# ============================================================
# PREPARAÇÃO
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

session = requests.Session()

session.headers.update({
    "User-Agent": "auditoria-calcadas-downloader/1.0"
})


# ============================================================
# CRIAR GRID
# ============================================================

def make_grid(bbox, nx=20, ny=20):
    """
    Divide a bounding box em pequenos retângulos.

    bbox:
        xmin, ymin, xmax, ymax
    """

    xmin, ymin, xmax, ymax = bbox

    dx = (xmax - xmin) / nx
    dy = (ymax - ymin) / ny

    tiles = []

    for i in range(nx):

        for j in range(ny):

            x1 = xmin + i * dx
            y1 = ymin + j * dy

            x2 = x1 + dx
            y2 = y1 + dy

            tiles.append(
                (x1, y1, x2, y2)
            )

    return tiles


# ============================================================
# CONSULTAR MAPILLARY
# ============================================================

def fetch_tile(tile):
    """
    Consulta todas as imagens de uma pequena região.
    """

    bbox_string = ",".join(map(str, tile))

    url = "https://graph.mapillary.com/images"

    params = {
        "fields": (
            "id,"
            "computed_geometry,"
            "thumb_original_url,"
            "captured_at,"
            "creator"
        ),

        "bbox": bbox_string,

        "creator_username": CREATOR_NAME,

        "start_captured_at": START_DATE,

        "end_captured_at": END_DATE,

        "limit": 1000,

        "access_token": ACCESS_TOKEN
    }

    imagens = []

    while url:

        try:

            response = session.get(
                url,
                params=params,
                timeout=TIMEOUT
            )

            response.raise_for_status()

            data = response.json()

        except requests.RequestException as e:

            print(
                f"\nErro consultando tile {tile}: {e}"
            )

            return imagens


        registros = data.get("data", [])

        imagens.extend(registros)


        # paginação

        paging = data.get("paging", {})

        url = paging.get("next")


        # A URL seguinte já contém os parâmetros,
        # portanto eles não devem ser enviados novamente

        params = None


        if url:
            time.sleep(0.25)


    return imagens


# ============================================================
# EXTRAIR LAT/LON
# ============================================================

def extrair_coordenadas(imagem):

    geometry = imagem.get("computed_geometry")

    if not geometry:
        return None, None

    coords = geometry.get("coordinates")

    if not coords or len(coords) < 2:
        return None, None

    lon = coords[0]
    lat = coords[1]

    return lat, lon


# ============================================================
# BUSCAR TODAS AS IMAGENS
# ============================================================

def buscar_imagens():

    tiles = make_grid(
        BBOX,
        GRID_X,
        GRID_Y
    )

    print(
        f"Área dividida em {len(tiles)} regiões."
    )

    todas = {}

    for numero, tile in enumerate(tiles, start=1):

        print(
            f"\rConsultando região "
            f"{numero}/{len(tiles)}",
            end=""
        )

        imagens = fetch_tile(tile)

        for imagem in imagens:

            image_id = str(
                imagem["id"]
            )

            # dicionário elimina duplicatas
            todas[image_id] = imagem


    print()

    print(
        f"{len(todas)} imagens únicas encontradas."
    )

    return list(todas.values())


# ============================================================
# CONVERTER PARA DATAFRAME
# ============================================================

def criar_dataframe(imagens):

    dados = []

    for imagem in imagens:

        creator = imagem.get(
            "creator",
            {}
        )

        creator_id = str(
            creator.get("id", "")
        )

        # mantém a mesma filtragem do código R
        if (
            TARGET_USER_ID
            and creator_id != TARGET_USER_ID
        ):
            continue


        lat, lon = extrair_coordenadas(
            imagem
        )


        captured_at = imagem.get(
            "captured_at"
        )

        if captured_at:

            captured_datetime = pd.to_datetime(
                captured_at,
                unit="ms",
                utc=True
            )

        else:

            captured_datetime = None


        dados.append({

            "id":
                str(imagem.get("id")),

            "creator_id":
                creator_id,

            "creator_username":
                creator.get("username"),

            "captured_at":
                captured_at,

            "captured_datetime":
                captured_datetime,

            "latitude":
                lat,

            "longitude":
                lon,

            "url":
                imagem.get(
                    "thumb_original_url"
                ),

        })


    df = pd.DataFrame(dados)

    if not df.empty:

        df = (
            df
            .drop_duplicates(
                subset=["id"]
            )
            .sort_values(
                "captured_datetime"
            )
            .reset_index(drop=True)
        )

    return df


# ============================================================
# DOWNLOAD DA IMAGEM
# ============================================================

def baixar_imagem(row):

    image_id = str(row["id"])

    url = row["url"]

    if not url:
        return image_id, "sem URL"


    destino = os.path.join(
        IMAGES_DIR,
        f"{image_id}.jpg"
    )


    # não baixa novamente
    if os.path.exists(destino):

        return image_id, "já existe"


    try:

        response = session.get(
            url,
            timeout=TIMEOUT
        )

        response.raise_for_status()


        with open(
            destino,
            "wb"
        ) as arquivo:

            arquivo.write(
                response.content
            )


        return image_id, "ok"


    except Exception as e:

        return (
            image_id,
            f"erro: {e}"
        )


# ============================================================
# DOWNLOAD PARALELO
# ============================================================

def baixar_todas(df):

    total = len(df)

    baixadas = 0
    existentes = 0
    erros = 0


    with ThreadPoolExecutor(
        max_workers=DOWNLOAD_WORKERS
    ) as executor:


        tarefas = {

            executor.submit(
                baixar_imagem,
                row
            ): row["id"]

            for _, row in df.iterrows()

        }


        for numero, future in enumerate(
            as_completed(tarefas),
            start=1
        ):

            image_id, status = (
                future.result()
            )


            if status == "ok":

                baixadas += 1


            elif status == "já existe":

                existentes += 1


            else:

                erros += 1

                print(
                    f"\n[ERRO] "
                    f"{image_id}: "
                    f"{status}"
                )


            print(

                f"\rDownload "
                f"{numero}/{total} "

                f"| novas: "
                f"{baixadas} "

                f"| existentes: "
                f"{existentes} "

                f"| erros: "
                f"{erros}",

                end=""

            )


    print()


# ============================================================
# MAIN
# ============================================================

# Token do Mapillary guardado nos Secrets do Colab (nome: Mapillary_api)
try:
    from google.colab import userdata
    ACCESS_TOKEN = userdata.get('Mapillary_api') or ACCESS_TOKEN
except Exception:
    pass

# Quantidade de fotos sorteadas para download (o projeto tem milhares)
MAX_IMAGENS = 300

def main():


    print(
        "\n=== AUDITORIA DE CALÇADAS ===\n"
    )


    print(
        "1. Buscando imagens no Mapillary..."
    )

    imagens = buscar_imagens()


    print(
        "\n2. Organizando metadados..."
    )

    df = criar_dataframe(
        imagens
    )


    print(
        f"{len(df)} imagens após filtragem."
    )

    # Amostra aleatória reprodutível para curadoria manual
    df = df.sample(n=min(MAX_IMAGENS, len(df)), random_state=42).reset_index(drop=True)
    print(f"Amostra para download: {len(df)} imagens.")


    # adiciona nome futuro do arquivo

    if not df.empty:

        df["arquivo"] = (
            df["id"]
            .astype(str)
            .apply(
                lambda x:
                os.path.join(
                    "imagens",
                    f"{x}.jpg"
                )
            )
        )


    # CSV
    df.to_csv(
        CSV_PATH,
        index=False,
        encoding="utf-8"
    )


    print(
        f"CSV salvo em: "
        f"{CSV_PATH}"
    )


    print(
        "\n3. Baixando imagens..."
    )


    baixar_todas(df)


    print(
        "\n=== FINALIZADO ==="
    )

    print(
        f"Imagens: "
        f"{IMAGES_DIR}"
    )

    print(
        f"Metadados: "
        f"{CSV_PATH}"
    )

    print(
        f"Total: "
        f"{len(df)} imagens"
    )


if __name__ == "__main__":

    main()

In [ ]:
import shutil
shutil.make_archive(base_name=OUTPUT_DIR, format='zip', root_dir=OUTPUT_DIR)

from google.colab import files
files.download(f"{OUTPUT_DIR}.zip")